# [B1/B3] Do STTR commercialization-channel rates differ by research-institution partner type?

**Status:** exploratory  
**Research question:** B1 — STTR research-institution partner types; B3 — Phase II→III
latency / transition ([docs/research-questions.md](../../docs/research-questions.md))
**Decision this informs:** whether a raw partner-type ranking on existing channels is
even visible before the gated STTR classifier is built. Not a freeze, not RQ2.
**Data as of:** whatever local artifacts are present (see input-status cell)  
**Owner:** STTR spinout-linkage workbench (PR #615)

Descriptive question only. Among STTR Phase II small businesses, how do *observed*
follow-on channels (coded Phase III, Form D, M&A) differ by a **coarse** RI-name
heuristic (`UNIVERSITY` / `FFRDC` / `COMMUNITY_COLLEGE` / `UNTYPED`)?

This is not a causal claim, not a matched comparison, and not the spec's partner-type
classifier. Spinout vs. subcontract labels do not exist and are not inferred here.
Do not collapse the three channels into one commercialization winner.

## Data contract

- **Population:** SBIR.gov awards with `program = STTR` and Phase II. Awards missing an
  RI name stay in the population and type as `UNTYPED`.
- **Grain:** award for partner type, agency, and vintage mix; firm (UEI, else
  normalized company name) for Form D and M&A. Phase III is joined at firm grain
  (any coded Phase II→III event for that UEI) and optionally at award grain when
  `phase_ii_award_id` matches.
- **Keys:** award `award_id` / Agency Tracking Number; firm `uei`; company-name join
  to Form D / M&A uses `CompanyNameProfile.FORM_D_JOIN_V1`. RI typing uses
  `CompanyNameProfile.MATCHING_V1`. No local normalizer.
- **Inputs:** first existing awards file among
  `data/processed/enriched_sbir_awards.parquet` and `data/raw/sbir/award_data.csv`;
  optional `data/processed/phase_transition_survival.parquet`,
  `data/processed/phase_ii_iii_pairs.parquet`, `data/form_d_details.jsonl`,
  `data/enriched_sbir_ma_events.jsonl`. Channel files are *searched* only when present.
- **Missingness:** a missing artifact means that channel was never searched here, not
  a negative. A missing RI name is typed absence (`UNTYPED`), not evidence the partner
  is a university or an FFRDC.
- **Outputs:** exploratory tables in this notebook. Non-citable. No canonical generator.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from sbir_etl.identity import CompanyNameProfile, normalize_company_name


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
REPO_ROOT

In [ ]:
AS_OF_DATE = "local-artifacts"  # replace with the awards file's as-of date when known
RANDOM_SEED = 20260814
NAME_PROFILE = CompanyNameProfile.MATCHING_V1
FORM_D_PROFILE = CompanyNameProfile.FORM_D_JOIN_V1

AWARD_CANDIDATES = (
    REPO_ROOT / "data" / "processed" / "enriched_sbir_awards.parquet",
    REPO_ROOT / "data" / "raw" / "sbir" / "award_data.csv",
)
REQUIRED_INPUTS: dict[str, Path] = {
    "awards": next((path for path in AWARD_CANDIDATES if path.exists()), AWARD_CANDIDATES[0]),
    "phase_iii_survival": REPO_ROOT / "data" / "processed" / "phase_transition_survival.parquet",
    "phase_iii_pairs": REPO_ROOT / "data" / "processed" / "phase_ii_iii_pairs.parquet",
    "form_d": REPO_ROOT / "data" / "form_d_details.jsonl",
    "ma_events": REPO_ROOT / "data" / "enriched_sbir_ma_events.jsonl",
}
GENERATORS = {
    "awards": "Dagster sbir ingestion / data/raw/sbir/award_data.csv",
    "phase_iii_survival": "transformed_phase_transition_survival",
    "phase_iii_pairs": "transformed_phase_ii_iii_pairs",
    "form_d": "scripts/archive/data/fetch_form_d_details.py",
    "ma_events": "historical M&A enrichment → data/enriched_sbir_ma_events.jsonl",
}

In [ ]:
input_status = pd.DataFrame(
    [
        {
            "input": name,
            "path": str(path.relative_to(REPO_ROOT)),
            "exists": path.exists(),
            "generator": GENERATORS[name],
        }
        for name, path in REQUIRED_INPUTS.items()
    ]
)
input_status

## Honesty block (read before any table)

1. **Heuristic types are not the spec classifier.** `UNIVERSITY` / `FFRDC` here are
   name-token labels. They are not IPEDS, not the NSF FFRDC Master List capture, and
   not a `CANDIDATE` assertion. Residual is `UNTYPED`, never `OTHER_NONPROFIT`.
2. **Unmatched.** Agency and vintage mix differ by partner type (NIH/NSF concentrate
   in universities). A higher university Phase III rate can be an agency effect.
3. **Coded Phase III is a lower bound** (Element 10Q undercount; see
   [phase-transition-latency.md](../../docs/phase-transition-latency.md)).
4. **A false channel flag is not a negative.** Missing Form D / M&A files mean those
   channels were not searched.
5. **No combined commercialization score.** Rank channels separately. [L47] finds
   Phase III receipt only weakly predictive of commercialization success.
6. **Not RQ2.** Spinout vs. subcontract is gated on frozen RQ1 labels.

## Sub-question 1 — STTR Phase II spine and coarse RI type

Filter to STTR Phase II. Normalize `ri_name` with `MATCHING_V1`. Assign `FFRDC` on a
short distinctive-name list (national labs / named FFRDCs), then `COMMUNITY_COLLEGE`,
then `UNIVERSITY` on university/college/institute-of-technology tokens, else `UNTYPED`.

In [ ]:
def first_col(frame: pd.DataFrame, names: tuple[str, ...]) -> str | None:
    lookup = {column.lower(): column for column in frame.columns}
    for name in names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    return None


def load_awards(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing {path.relative_to(REPO_ROOT)} — run {GENERATORS['awards']}.")
        return pd.DataFrame()
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype=str, low_memory=False)


def is_sttr(value: object) -> bool:
    text = str(value or "").strip().upper()
    return text == "STTR"


def is_phase_ii(value: object) -> bool:
    text = str(value or "").strip().upper().replace("PHASE ", "")
    return text in {"II", "2"}


awards_raw = load_awards(REQUIRED_INPUTS["awards"])
if awards_raw.empty:
    sttr_p2 = pd.DataFrame()
else:
    program_col = first_col(awards_raw, ("program", "Program"))
    phase_col = first_col(awards_raw, ("phase", "Phase"))
    if program_col is None or phase_col is None:
        print(f"Awards file lacks program/phase columns: {list(awards_raw.columns)}")
        sttr_p2 = pd.DataFrame()
    else:
        sttr_p2 = awards_raw.loc[
            awards_raw[program_col].map(is_sttr) & awards_raw[phase_col].map(is_phase_ii)
        ].copy()
        print(f"STTR Phase II award rows: {len(sttr_p2):,}")
sttr_p2.head() if not sttr_p2.empty else sttr_p2

In [ ]:
FFRDC_PHRASES = (
    "lincoln laboratory",
    "mitre",
    "aerospace corporation",
    "sandia",
    "los alamos",
    "lawrence livermore",
    "lawrence berkeley",
    "argonne",
    "oak ridge",
    "pacific northwest national",
    "brookhaven",
    "idaho national",
    "national renewable energy",
    "fermi",
    "slac",
    "jefferson lab",
    "ames laboratory",
    "princeton plasma",
    "software engineering institute",
    "atmospheric research",
    "jet propulsion",
    "savannah river",
    "institute for defense analyses",
    "national radio astronomy",
)
UNIVERSITY_TOKENS = frozenset(
    {
        "university",
        "universidad",
        "universite",
        "universitat",
        "college",
        "colleges",
        "polytechnic",
    }
)


def coarse_partner_type(ri_name: object) -> str:
    normalized = normalize_company_name(ri_name, profile=NAME_PROFILE)
    if not normalized:
        return "UNTYPED"
    if any(phrase in normalized for phrase in FFRDC_PHRASES):
        return "FFRDC"
    if "community college" in normalized:
        return "COMMUNITY_COLLEGE"
    tokens = set(normalized.split())
    if "institute of technology" in normalized or tokens & UNIVERSITY_TOKENS:
        return "UNIVERSITY"
    return "UNTYPED"


if sttr_p2.empty:
    typed = pd.DataFrame()
else:
    ri_col = first_col(sttr_p2, ("ri_name", "RI Name", "research_institution"))
    agency_col = first_col(sttr_p2, ("agency", "Agency"))
    year_col = first_col(sttr_p2, ("award_year", "Award Year"))
    date_col = first_col(sttr_p2, ("award_date", "Proposal Award Date"))
    uei_col = first_col(sttr_p2, ("uei", "UEI", "company_uei"))
    company_col = first_col(sttr_p2, ("company_name", "Company", "company"))
    award_id_col = first_col(
        sttr_p2, ("award_id", "Agency Tracking Number", "agency_tracking_number")
    )
    typed = pd.DataFrame(
        {
            "award_id": sttr_p2[award_id_col] if award_id_col else pd.NA,
            "agency": sttr_p2[agency_col] if agency_col else pd.NA,
            "award_year": sttr_p2[year_col] if year_col else pd.NA,
            "award_date": sttr_p2[date_col] if date_col else pd.NA,
            "uei": sttr_p2[uei_col] if uei_col else pd.NA,
            "company_name": sttr_p2[company_col] if company_col else pd.NA,
            "ri_name": sttr_p2[ri_col] if ri_col else pd.NA,
        }
    )
    typed["partner_type"] = typed["ri_name"].map(coarse_partner_type)
    typed["firm_key"] = typed["uei"].fillna("").astype(str).str.strip().str.upper()
    missing_uei = typed["firm_key"].eq("") | typed["firm_key"].isin(
        {"NAN", "NONE", "NULL", "<NA>"}
    )
    typed.loc[missing_uei, "firm_key"] = typed.loc[missing_uei, "company_name"].map(
        lambda value: normalize_company_name(value, profile=FORM_D_PROFILE)
    )
    unresolvable_firm_key = typed["firm_key"].astype(str).str.len().eq(0)
    n_unresolvable = int(unresolvable_firm_key.sum())
    if n_unresolvable:
        print(
            f"{n_unresolvable:,} STTR Phase II awards have neither a UEI nor a company name; "
            "kept in the population under a per-award sentinel firm_key so they still count "
            "toward partner-type totals, but each is its own unjoinable 'firm' downstream."
        )
    typed.loc[unresolvable_firm_key, "firm_key"] = (
        "UNRESOLVED_FIRM_" + typed.loc[unresolvable_firm_key].index.astype(str)
    )
    print(typed["partner_type"].value_counts(dropna=False).rename("awards").to_string())
typed.head() if not typed.empty else typed

## Sub-question 2 — agency and vintage mix

A raw ranking that ignores agency and award year is not interpretable. Universities
should dominate NIH/NSF; FFRDCs should concentrate in DoD/DOE if the heuristic fires.

In [ ]:
if typed.empty:
    agency_mix = pd.DataFrame()
    vintage_mix = pd.DataFrame()
else:
    # pd.crosstab counts rows (like aggfunc="size"), not non-null values of one column
    # (like pivot_table's aggfunc="count"), so an award with a blank award_id still counts.
    agency_mix = pd.crosstab(
        typed["partner_type"],
        typed["agency"].fillna("(missing)").astype(str).str.strip(),
        margins=True,
    )
    year = pd.to_numeric(typed["award_year"], errors="coerce")
    if year.isna().all() and "award_date" in typed.columns:
        year = pd.to_datetime(typed["award_date"], errors="coerce").dt.year
    vintage = typed.assign(award_year=year).dropna(subset=["award_year"])
    decade = (vintage["award_year"].astype(int) // 10) * 10
    vintage_mix = pd.crosstab(vintage["partner_type"], decade, margins=True)
agency_mix

In [ ]:
vintage_mix

## Sub-question 3 — channel observation rates by partner type

Collapse to firm grain (one partner type per firm: unique type if unanimous, else
`MIXED`). Join coded Phase III (UEI on the survival/pairs artifacts), high-confidence
Form D names, and high/medium M&A names. Report each channel separately, then the
same rates inside agency slices so mix cannot hide as a type effect.

In [ ]:
def load_jsonl_names(path: Path, *, keep) -> set[str]:
    names: set[str] = set()
    if not path.exists():
        print(f"Missing {path.relative_to(REPO_ROOT)} — channel not searched.")
        return names
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            if not keep(record):
                continue
            name = record.get("company_name")
            key = normalize_company_name(name, profile=FORM_D_PROFILE)
            if key:
                names.add(key)
    print(f"{path.name}: {len(names):,} joinable company names")
    return names


def phase_iii_ueis() -> set[str]:
    found: set[str] = set()
    survival = REQUIRED_INPUTS["phase_iii_survival"]
    pairs = REQUIRED_INPUTS["phase_iii_pairs"]
    if survival.exists():
        frame = pd.read_parquet(survival)
        uei_col = first_col(frame, ("recipient_uei", "uei"))
        event_col = first_col(frame, ("event_observed",))
        if uei_col and event_col:
            observed = frame.loc[frame[event_col].fillna(False).astype(bool), uei_col]
            found.update(observed.dropna().astype(str).str.strip().str.upper())
        print(f"survival Phase III UEIs: {len(found):,}")
    elif pairs.exists():
        frame = pd.read_parquet(pairs)
        uei_col = first_col(frame, ("recipient_uei", "uei"))
        if uei_col:
            found.update(frame[uei_col].dropna().astype(str).str.strip().str.upper())
        print(f"pairs Phase III UEIs: {len(found):,}")
    else:
        print("No Phase III survival/pairs artifact — channel not searched.")
    return found


if typed.empty:
    firms = pd.DataFrame()
else:
    type_nunique = typed.groupby("firm_key")["partner_type"].nunique()
    primary = typed.groupby("firm_key")["partner_type"].agg(
        lambda series: series.mode().iloc[0] if not series.mode().empty else "UNTYPED"
    )
    firms = (
        typed.groupby("firm_key", as_index=False)
        .agg(
            n_sttr_p2_awards=("award_id", "size"),
            agencies=("agency", lambda s: ",".join(sorted({str(v) for v in s.dropna()}))),
            primary_agency=("agency", lambda s: s.mode().iloc[0] if not s.mode().empty else "(missing)"),
            company_name=("company_name", "first"),
            uei=("uei", "first"),
        )
        .assign(
            partner_type=lambda frame: frame["firm_key"].map(primary),
            mixed_partner_type=lambda frame: frame["firm_key"].map(type_nunique).gt(1),
        )
    )
    firms.loc[firms["mixed_partner_type"], "partner_type"] = "MIXED"
    firms["name_key"] = firms["company_name"].map(
        lambda value: normalize_company_name(value, profile=FORM_D_PROFILE)
    )
    phase_iii = phase_iii_ueis()
    form_d_names = load_jsonl_names(
        REQUIRED_INPUTS["form_d"],
        keep=lambda rec: (rec.get("match_confidence") or {}).get("tier") == "high",
    )
    ma_names = load_jsonl_names(
        REQUIRED_INPUTS["ma_events"],
        keep=lambda rec: rec.get("confidence") in {"high", "medium"},
    )
    uei_key = firms["uei"].fillna("").astype(str).str.strip().str.upper()
    firms["phase_iii_observed"] = uei_key.isin(phase_iii)
    firms["form_d_observed"] = firms["name_key"].isin(form_d_names)
    firms["ma_observed"] = firms["name_key"].isin(ma_names)
    searched = {
        "phase_iii": REQUIRED_INPUTS["phase_iii_survival"].exists()
        or REQUIRED_INPUTS["phase_iii_pairs"].exists(),
        "form_d": REQUIRED_INPUTS["form_d"].exists(),
        "ma": REQUIRED_INPUTS["ma_events"].exists(),
    }
    print("channels searched:", searched)
    print(f"STTR Phase II firms: {len(firms):,}")
firms.head() if not firms.empty else firms

In [ ]:
def rate_table(frame: pd.DataFrame, *, by: list[str], searched: dict[str, bool]) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()
    grouped = frame.groupby(by, dropna=False)
    out = grouped.agg(
        n_firms=("firm_key", "nunique"),
        n_phase_iii=("phase_iii_observed", "sum"),
        n_form_d=("form_d_observed", "sum"),
        n_ma=("ma_observed", "sum"),
    )
    for src, count in (
        ("phase_iii", "n_phase_iii"),
        ("form_d", "n_form_d"),
        ("ma", "n_ma"),
    ):
        out[f"{src}_rate"] = (out[count] / out["n_firms"]).where(out["n_firms"] > 0)
        if not searched.get(src, False):
            # Channel file absent — an unsearched channel must never read as a
            # measured 0.0 (see honesty rule 4 and the data-contract missingness note).
            out[count] = pd.NA
            out[f"{src}_rate"] = pd.NA
    return out.sort_values("n_firms", ascending=False)


by_type = rate_table(firms, by=["partner_type"], searched=searched)
by_type

In [ ]:
by_type_agency = rate_table(firms, by=["partner_type", "primary_agency"], searched=searched)
by_type_agency

## Findings and caveats

| Claim | Evidence/artifact | Caveat or alternative explanation |
|---|---|---|
| _Draft — no ranking asserted until awards and at least one channel file are present_ | `by_type` / `by_type_agency` | Agency mix can produce a university or FFRDC "win" without a partner-type effect |
| Heuristic `UNTYPED` share is the coverage floor of this probe | partner-type value counts | Hospitals, FFRDCs not on the short list, and new-model orgs all land here |
| Spinout vs. subcontract is not measured | spec RQ1/RQ2 gated | Do not read partner type as a spinout proxy |

If a type×channel cell looks large enough to brief, the next step is still not a
citation: it is either a matched notebook (agency × year) or waiting for frozen RQ1
labels. This notebook remains exploratory and non-citable.

## Promotion checklist

- [x] Question and decision are explicit.
- [x] Data snapshot, population, grain, keys, and exclusions are recorded.
- [x] Samples and stochastic methods use a deterministic seed.
- [x] The destination tier and its contract are explicit (`exploratory`, non-citable).
- [ ] Recurring calculations have one canonical implementation.
- [ ] Recurring artifact generation has a thin CLI or Dagster asset.
- [ ] Citable work satisfies all four `evidence` contract requirements.
- [ ] Findings and methodology are linked from `docs/`.
- [x] Outputs and execution counts are cleared before commit.

Until promotion is complete, this notebook and its claims remain exploratory and non-citable.